# ABCI-MI — Full-Application Kaggle REAL Execution

This notebook runs the **existing backend production CLI**: meeting service →
LongAudioProcessor/MOSS → PyAnnote/Sarvam → reconstruction → ACE/Blackboard →
SKW/PostgreSQL/Redis/Qdrant. Notebook code only configures, launches and observes
the repository. The embedded source snapshot contains current uncommitted fixes;
its archive/file SHA-256 manifest identifies the code actually executed.

**Before Run All:** enable a Kaggle GPU and Internet, attach the actual full audio
as an input, and configure Secrets `HF_TOKEN`, `DATABASE_URL_ASYNC`, `REDIS_URL`,
`QDRANT_URL` (plus `QDRANT_API_KEY`/`ABCI_USER_ID`/`ABCI_TENANT_ID` if needed).
The database must already have this repository's schema and the authorized CLI
user. PostgreSQL/Qdrant must be reachable from Kaggle. A Windows localhost URL
does not reach the laptop from Kaggle. Redis's existing fallback is reported
explicitly; it prevents a distributed-path VERIFIED classification.

Optional environment/Secret `ABCI_AUDIO_PATH` selects an existing mounted file;
otherwise ES2002a.Mix-Headset.wav is preferred, or the sole attached audio file.
Attach existing HF model caches to avoid duplicate model downloads. HF gated
model access must already be granted. Missing files alone are provisioned using
the repository's official model provisioning script. No datasets are downloaded.

No cell edits are required. Run All records BLOCKED rather than fabricating
results when prerequisites fail. The notebook is **not executed** in this saved
deliverable. Reference `sucess1.ipynb` is short Kaggle evidence, not full-application
or local validation. No tokens, original reference outputs or local .env files
are embedded. Raw real transcript evidence is saved privately under working output;
review it before sharing an executed notebook/output dataset.

## 1–2. Environment and repository setup
The backend snapshot is embedded to preserve actual current changes. No Git credentials or local configuration are included.

In [ ]:
import base64, hashlib, io, json, os, re, shutil, subprocess, sys, time, uuid, zipfile
from pathlib import Path
from datetime import datetime, timezone
RUN_ID = "kaggle_real_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:12]
WORK = Path("/kaggle/working") / RUN_ID
WORK.mkdir(parents=True, exist_ok=False)
REPO = WORK / "application"
EVIDENCE = WORK / "evidence"
EVIDENCE.mkdir()
STAGES = {}
BLOCKERS = []
def save(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    candidate = path
    version = 1
    while candidate.exists():
        candidate = path.with_name(f"{path.stem}_{version:03d}{path.suffix}")
        version += 1
    with candidate.open("x", encoding="utf-8") as f: json.dump(value, f, indent=2, default=str)
    return candidate
def redact(value):
    text = str(value)
    for key, secret in os.environ.items():
        if secret and any(x in key for x in ("TOKEN", "PASSWORD", "SECRET", "URL", "API_KEY")):
            text = text.replace(secret, "[REDACTED]")
    text = re.sub(r"hf_[A-Za-z0-9]{12,}", "[REDACTED]", text)
    return re.sub(r"(://)[^/@\s]+:[^/@\s]+@", r"\1[REDACTED]@", text)
def gated(name, function, depends=()):
    if any(STAGES.get(dep, {}).get("status") != "VERIFIED" for dep in depends):
        result = {"status": "BLOCKED", "reason": "Prerequisite gate did not pass", "dependencies": list(depends)}
    else:
        try: result = function() or {"status": "VERIFIED"}
        except Exception as exc: result = {"status": "BLOCKED", "error_type": type(exc).__name__, "reason": redact(exc)}
    STAGES[name] = result
    save(EVIDENCE / f"{name}.json", result)
    if result["status"] == "BLOCKED": BLOCKERS.append({"stage": name, **result})
    print(name, result["status"])
    return result


In [ ]:
SOURCE_COMMIT = 'bd31603dfe02ce0c82020301aa5307eee9070bd1'
SOURCE_ARCHIVE_SHA256 = '3d81918efab97903994aa8d3bb43529d199a41370f24bb304e4476eeb4b057b5'
SOURCE_MANIFEST = {'backend/app/__init__.py': '8da907af5aaf3524c76dda0e2e19cfca3f87ed09a2404caa6e6c99e5b35ff083', 'backend/app/ai/__init__.py': 'c755790916c86dcf9f21c51f4ece8e8dcbb157945aac67bb87dfb8b3835a49c8', 'backend/app/ai/code_switch_intelligence.py': '18f32e776140377d23ef8c49afb053289daf2e95c2f555993578fa87409d13f6', 'backend/app/ai/confidence_fusion.py': '9f1b93bccde2fa441f39a57ada027a81851eb063f8451e8ddd86c7c339281613', 'backend/app/ai/context_intelligence.py': 'af6e895bc44a3310f8c4a1f6e6d04899d9a2c5d64f04f263c5e5d090b03df353', 'backend/app/ai/knowledge_memory.py': '730477047764f5fe273df58cf1d325f4abc9152b545336178e17e62a23cced49', 'backend/app/ai/long_audio_processor.py': '7aba04ea2d2944d2f31121904e0c80959f15c13fceb62b702fd25c17f5cd701e', 'backend/app/ai/meeting_analytics.py': 'cb76c0d358d3c6e5714826dc6450f6e7602d6d11149b5c2e50a84b3903581523', 'backend/app/ai/meeting_understanding.py': 'f04dd9072769e3f9cc59d080e904fa57e5b31ee5e4326724d2f68d310584648e', 'backend/app/ai/model_inventory.py': '64d4d625d266fd486da42145ee85c91b41f12f076888aa17115581adcfd3194d', 'backend/app/ai/model_verification.py': '7ca12879e9a8af47de9ac8ab18de2e8de2d41c13eb2299243247fd8926882a12', 'backend/app/ai/moss_adapter.py': '91b91f0af78c98ce947ec510d7ad7f3cc74403f969e1dd022acdf9e795d1744d', 'backend/app/ai/multilingual_asr.py': '91c134cf15132f9e1ed82185ca189ce9940de2a994574b32a632fda0d7febedd', 'backend/app/ai/overlap_resolution.py': 'e5e0ceafc7056df98d9527ad6e11ef3b5f172e44d329015e780656c3eb872ef7', 'backend/app/ai/real_validation.py': '5f5a9cfe2e93c2c4d6fe0e8dc55ad259b4096ee4a451dc01e5dbefdb29492e5b', 'backend/app/ai/speaker_diarization.py': 'bb4a9baf9b6d3655e9c8df3469893c7a73acdc202de54189853495107a333040', 'backend/app/ai/transcript_intelligence.py': '5ab746d401d62f50aacec2146749e23d76969b730107018a24f6ed43e018d45e', 'backend/app/ai/translation_engine.py': 'c9a9a80f3b0e9cf863e87b08a0436b15713a8e56dcc5c501b3135f3f226849c7', 'backend/app/ai/verification_engine.py': '06b8226e57c5312c227dd970fa0e82e047e6390225080eea79ab9d1c5b7b653f', 'backend/app/api/__init__.py': '437d341be7085c69843b393c5d6522d00032445f8dfc39c5da5373835dd8abf5', 'backend/app/api/dependencies.py': '847cd967193c7bd3af0cc2ef6d33dd345be04741d024a4a475a47fa3c9b8a527', 'backend/app/api/v1/__init__.py': 'a8b15ecc07b618b0e728fc1f55a0271fe0bd6558804d3076e968f5e088e40bbf', 'backend/app/api/v1/endpoints/__init__.py': 'b20f59b7722e1720bddc49fa863b1cc798a5eca511b8b9c0835cb8379bba06dc', 'backend/app/api/v1/endpoints/action_items.py': '1da4697b9c4b3c35305492c2073334db1a63521de116220c50af52da7f739cfa', 'backend/app/api/v1/endpoints/admin.py': '616743f5e1e4995f7db819c389763905d1ab8eb840f1e1b1d742d8b3d824e40d', 'backend/app/api/v1/endpoints/auth.py': '4b5f259a65004455f1289f0ed7e2ca5c49daf6de50952086fa29b6fc221cb46c', 'backend/app/api/v1/endpoints/health.py': 'bcbc3239f045d163099066efa776c09698bee8f81fd36deebd6fb4d28c96079a', 'backend/app/api/v1/endpoints/knowledge.py': 'e03634d7a19f5a1a4deefd1f846010ccbc6715350188b629ec16561beb8c1da4', 'backend/app/api/v1/endpoints/live_sessions.py': 'ddd7fdfcdc170f38eafbe8ac7952396057a02a75f51e701cd64c8f92894d99e0', 'backend/app/api/v1/endpoints/meeting_intelligence.py': 'aa716889f4723b1418d623d562ad18e41be13172e88431ba8dbd85ab8fadae83', 'backend/app/api/v1/endpoints/meetings.py': '78ef9179e614d37c2ecbfbb36b2d50e64d146aade7b87e1f2a7d73bfada86152', 'backend/app/api/v1/endpoints/memory.py': '17aada6cc5bf8dc646eeb11534823d594f0fd48c530f10f831a1f9e14828d4d4', 'backend/app/api/v1/endpoints/notifications.py': '57f6dee3a73d687d5ae4155af4926a16e22e1ed1506ecb5fab60be58d3acfedf', 'backend/app/api/v1/endpoints/platform_integrations.py': '5058cccc7863245ae9ddbd7d9b8698505a17a556a59621bc11dfb1d13dcfad57', 'backend/app/api/v1/endpoints/projects.py': 'b0363b7851999fc23d0bf33d4edc5c1ea6132f63f0a9d5ff8ebcaa31b7b73c80', 'backend/app/api/v1/endpoints/query.py': 'ea267ed0e94405e82ded4eb983d00cf864452fddae187fc32f41005012f20111', 'backend/app/api/v1/endpoints/reports.py': 'a0ea110c4df8a8f3c079c41e036a5bb5add79138bf8430c1d942de711fd3ab5a', 'backend/app/api/v1/endpoints/translations.py': '3ea0d5b7754f10bf81456b737fbe0f920ff4c2184c9d328dc8a036d24c9dcdc5', 'backend/app/api/v1/router.py': 'a98e9b1f6e200f16bb88951acb32e7993c3278410f985e08963f3bf33125bea5', 'backend/app/benchmarks/__init__.py': 'f94005bf2f463eeb112bf231ecb012c9a7a75e9068f653b678e23cb2d2189c1a', 'backend/app/benchmarks/baseline.py': '4723343f32345c830e6134c365c4be050cfdbf0a4734036d469f0e8df89fe377', 'backend/app/benchmarks/benchmark_cli.py': '738505de03a45841c2414709b63a73c86fc1f69cb179b080b0c66e68211f5e49', 'backend/app/benchmarks/benchmarkcli.py': '9b935584366358874924d202d8004f49d22fe8c50f1340e2cc8e4b9ef11bc16c', 'backend/app/benchmarks/cli.py': '8f7e0cf550a19906626194cf26c62a90fd55a3663377af5ced2e8454376e91a7', 'backend/app/benchmarks/config.py': '1030d497ea7416bbbb1edc963f4f1a68d8cb72c232ac43b6fbcdffb9e9a59094', 'backend/app/benchmarks/dataset_registry.py': 'c65746250279f462c369f01a4ce4464d5d0250a0b4267bac8c6cac19170cb813', 'backend/app/benchmarks/datasets/__init__.py': '903d9637cd63a0886956d380e7bba8014ea357c223064b54aaf5d15924126fd5', 'backend/app/benchmarks/datasets/aishell.py': '4feb4e868b8fdfc0ba971751f2f4b6f415fe5aa4b87dfa46ed89e2f957c8307f', 'backend/app/benchmarks/datasets/ami.py': '0c67d529423567849d740ec6c2fede47fcf34767f04680cdd5c30bf9583d18f1', 'backend/app/benchmarks/datasets/base.py': '78da9663f3a0fc2b7b8f25f2ccdf13a72dcaf2cf6d93b328660262c3abd2188d', 'backend/app/benchmarks/datasets/common_voice.py': '9ea0bd5a704c960b803b24976c17d2db17a5e69e26222dff47d22a344cf1af31', 'backend/app/benchmarks/datasets/dihard.py': 'aea984441b5ff9756acf455421c3d7b1bd70b08886b198f3bedbf1de6280d127', 'backend/app/benchmarks/datasets/fleurs.py': '5a2cff34ac2a7f2a87426b605dd54efec5cbebc505512c82f8d42c57892d1d3b', 'backend/app/benchmarks/datasets/indicsuperb.py': '3bdc6048188b7b5484ac0259c24d294cf71d0813ea211343e1f038f7e4f48725', 'backend/app/benchmarks/datasets/mucs.py': '4a36b1fbdc6a4ae4e954af86154277a66c283f8eb8c2243d701611e848510f3b', 'backend/app/benchmarks/datasets/voxconverse.py': '52b18368e5699ba7a70645640a7ec8a865ba961dbde274e6281993a3c0ffef84', 'backend/app/benchmarks/exceptions.py': '68dd6e8b402c30ce883d42ece6e7df706b5c60ab8436dbcf1bc0bf3f0ab18aa7', 'backend/app/benchmarks/metrics.py': '2660ecdcfb9375b6a9a4b6c329547663d9c77ae624e226039c978dac286c3f88', 'backend/app/benchmarks/reporting.py': '51c6d937fb100ff25508bf85d308ffe58f49c464ed7671ac12af3524d002eb70', 'backend/app/benchmarks/runner.py': '3823eeff93e91e2a8468fa88784ed280cd9c38a33db60ed1a40dcf6b84c30263', 'backend/app/benchmarks/storage.py': '5cfef22ef79cdaf64e74568bd969a863ee5c58d4dce4259e41b393d8984eaff8', 'backend/app/cli.py': '2749298aba5f152f8047cd32bf32a5f3896b7ca40b56a9e3e4e4ae95e6cdaa6b', 'backend/app/core/__init__.py': 'c36517bd6a5e9b460846dfa8e5db59c714b8cdc30ba0ab70851e3016cbeb4893', 'backend/app/core/config.py': '5a5d4a052d3336b00bb82365c74c7778217d00242fc545978fc53200bc3891fb', 'backend/app/core/exceptions.py': '3e1f2b2d537419a7377129445bc030feaf964c94c6995463f15a0c2db60703d5', 'backend/app/core/logging.py': '23ac075cbeed853867c2510d835d5441096bff695d358fee2da041354b309801', 'backend/app/core/security.py': 'a172654f524d0587939159b8890e5f731c16db7ca34b78bfd31dd4aa272f507c', 'backend/app/events/__init__.py': '4de0b97592498e74014d2d8f3b518a4ec08f456048da5059d80dc8c1cb840010', 'backend/app/events/redis_bus.py': 'c1b605db4513c171b6fc0e3e3224cfa3f877dfe9b397eb7823aa60800a547072', 'backend/app/infrastructure/__init__.py': 'bcc31f732b2d47309d44af11f20898bb70ef69e53f8368ae35777dc8518f6451', 'backend/app/infrastructure/database.py': '11dcc0c82b7e52138fb97c293a7dfa3e3651a2dc3c5c07dd66f8af74c041deaa', 'backend/app/infrastructure/qdrant.py': 'a120ed4d2396f44141a766839e946b1821ab2a5cd6dadb36498ff071ec5c0d25', 'backend/app/infrastructure/redis.py': '5ae80482556828df9ad8c69e11c678d6c57896d8a3a282b601109a3b675b19ad', 'backend/app/infrastructure/storage.py': '2099cbaf1f5e7c7afcc795e1d05c951fcf3df2c5d33ff04107383cd8290c86cb', 'backend/app/knowledge/__init__.py': 'e0bf10ca99304dd041b9c0fdfbcc15dac3608a645413e57e43a57a6843c9ace9', 'backend/app/knowledge/knowledge_service.py': '974d1cef30b43e977c6015c26750437c176f9387849d749fce1026dda6c8a0d9', 'backend/app/main.py': '2eb3c3045df51243d89c91717dee55fb43ba39cd9265f98ad1783495e79eea64', 'backend/app/models/__init__.py': 'f3f9dfc0eb47ce3f3af0adae1a4a03de765b5b47cb414f30def4afecb61d446e', 'backend/app/models/analytics.py': '732eddd15b1fcf6606c5aac9f445448ba143f669c11343d618b5ba15349e118c', 'backend/app/models/audio.py': '03c9ace9d1aa28032426d66f502eb0695a8efdf96f0e7c30bf9f46c32eed0ea5', 'backend/app/models/audit_log.py': 'b1a6fa763d5d3b919fe3652dc1111569d760b03769adde2030448c0858ef02cc', 'backend/app/models/base.py': '8f0767f6933425930264164e950fbd6dc5a75df1112565f71618b78885336d59', 'backend/app/models/configuration.py': '631a496a53a857239adf0636c6df4185a8378d5b916b4e334331ee23ad5f53bb', 'backend/app/models/knowledge_object.py': '67552d71350d8c44b157ecfef3c60664e7884f1e56366ee285e745f163ca2e93', 'backend/app/models/live_session.py': 'f0efe3e362cdab96e4f986f53309f6d288271bbe62f63fda90af1f99336d9348', 'backend/app/models/meeting.py': '6404d95adf957357e8de805c3b0958f8acc1e3d41b05ab1ba00222b7c0e78997', 'backend/app/models/notification.py': '10a5d58efff9d5db634448d6dcc1ece36ee10d927996bd0670e36f90d65b5d6d', 'backend/app/models/participant.py': '6c5dcdf8992950a50b1de2ebdf04a2d2ed0e64f292a5cac6fe6c75e2cfb6cb88', 'backend/app/models/platform_integration.py': 'b4dc89f3a2a0615faf95f9262fd425a1a6837ca74f916c306055a2c370be18be', 'backend/app/models/project.py': '46757bf78a17bd6a89c68e71147074db372179478daaf9e360e1d6cf38f90101', 'backend/app/models/query_record.py': '5d668a0e283c36a1f7a68a2901b8752af1484222b89f0ae0a7a0ed07cdff21cd', 'backend/app/models/report.py': 'cae03c913356d737cf93a12d13fdc78660e852759dacd8c794082a2d0ee308fc', 'backend/app/models/transcript.py': '529cd7858987bda1b22a596966bf69a0ff168e37cced7c4fd508d02058c59f4e', 'backend/app/models/translation.py': '959fd99a678f33231dd81249d6e16eb2bb8547ce4c1eb4dd18b2ced323a52d3a', 'backend/app/models/user.py': '425404d81327b4dff6d9e086529dc3590e7922f5f0e2013d96e539094d1fd4c6', 'backend/app/orchestration/__init__.py': 'e5ce710c528c79f21936a8351752a42d8d708b730dcf6ee28bbad615567cf79d', 'backend/app/orchestration/ace_boundary.py': '7e1d397db4ea3ca8ced68d93780adb81817a0b828856c4adea0ca7c1c109811f', 'backend/app/orchestration/ace_core.py': 'cd8433a8b4a2d557ec37bef524fc9f066bd9864cced24ae1dde0d84f5450a3f0', 'backend/app/orchestration/ace_engine.py': '1405bc9699c6b0cde271c477793ddfb4e2e7f1da79c8bcf563112885b85a1da6', 'backend/app/orchestration/blackboard_context.py': '4d390ba33858b96bed7ea3f9c2e5c293c32600f0b37663b494d772d6436e24ca', 'backend/app/orchestration/blackboard_event_consumer.py': 'a0923874faa0f0db2a81053113ad84ef0814487bef4c7190659b75be6ca4d30c', 'backend/app/orchestration/events.py': 'e827f0a578cefaf0b48fa14dc201e83664fcdb81fce0954380866ee0ac91474b', 'backend/app/orchestration/module_runner.py': '8d826ce894b7cc611228dcecff0b6cb85ed4c5372c88725a9b8896ba03a9e9ae', 'backend/app/orchestration/recovery_manager.py': '5da1c6a7637e780f4983a690f7bb36a2108b38168a15fe48edf1bbad7660b6fa', 'backend/app/orchestration/skw_client.py': 'daed458c536f05231c6ee46dcefd102e404925bc156530e0a469510974844146', 'backend/app/repositories/__init__.py': 'f4f8ac25852676bea670b038f04582218337bebe0db70a312399d90e2975ee24', 'backend/app/repositories/analytics_repo.py': '58dff4c7a77f7d14704734da596b951e6ba6247dec28234981891757622edc59', 'backend/app/repositories/audio_repo.py': '5f78ee2cc4508d615b840654f8cbfc9ba24d90b60104ad49ec6db8bdc1148439', 'backend/app/repositories/audit_log_repo.py': '46bf5506ba2357842b5d563fee91b946a3517ca9df222929a9ce63fe0a48a705', 'backend/app/repositories/audit_repo.py': '57d3334f978c4d72c1fb190a9d14182cb07dbb5887d784e35ea945fe034ec106', 'backend/app/repositories/base.py': 'bb37749b4ad20dcbb1027caaac9169e79adbcdb4352c7be0ab6f52e94fdfb4ab', 'backend/app/repositories/configuration_repo.py': 'fd45dad4efe85357190887f6c220f456e34ee68f449796a10cefea8f4bb3beef', 'backend/app/repositories/knowledge_object_repo.py': '0246ffb9be19b3688af8e2684da77f44342c6c367ef8bbfd933c40dd7f321dad', 'backend/app/repositories/meeting_repo.py': '7f6e76b389110db8b6c7963287a9bec47cf493fff61f80a52366ccde527582bb', 'backend/app/repositories/notification_repo.py': 'ba079735318c07cf0350be780c8ebe289ff8bcecba443caf0b2f9e91448798cb', 'backend/app/repositories/participant_repo.py': '319e4c368734e4d3ce9db14c82947d48fcc4c7ebbc990c6a962fdf3a361875ed', 'backend/app/repositories/project_repo.py': '69457c083f78de2e171c034d5a1692d92ec4a5de433e0d7cf98cd418b9a4151f', 'backend/app/repositories/query_repo.py': '8c297bd1dee75f71b1a29a827383eed7f5778d382f167e8f2ebdda581499a81d', 'backend/app/repositories/transcript_repo.py': 'c1bc84354b77e0b549f139a0a9933ace50615b0be64363b5ae7a24a4aa4bbb32', 'backend/app/repositories/user_repo.py': 'c7d2043bc8a8c25084986017f8ceb52eb20c18ee011526f6a17c3f482b2c0019', 'backend/app/schemas/__init__.py': 'a203ce76c493085c487676ccafe98aec51ae6358b59e45c620a14b4c30807c6c', 'backend/app/schemas/action_item.py': '9e4d4e08600f6e63c611824358ac791e9e71f8823673c110fc515d6b41b8b674', 'backend/app/schemas/admin_audit.py': 'b62dd64885aa19732a2eb24a1ce667a7215c33bfe6e3664b6c10ee9423fa5f50', 'backend/app/schemas/base.py': 'a55d5519c2547cdcc0e4d816aa254a0d5cf2fd89ef0b525bc6d9b71dc2d9060b', 'backend/app/schemas/health.py': '3a4ec52c3ef48cc1d0158dbba56d794afac2edac4d705602299300c4dfd5fc71', 'backend/app/schemas/knowledge_object.py': 'a50baa206b56eb991bd2ce292d6a564ee036b5936fae7474416335b4a026e5b7', 'backend/app/schemas/live_session.py': 'ea9469722c07f9fbd5a22b644b6bc7a88bdc3b8cc703cd1ff9a544c178ae99f9', 'backend/app/schemas/meeting.py': 'a3f42b46c10985583db9bae9ac3cb956ce2fd54c80e493dbcb08aa639448fbe8', 'backend/app/schemas/meeting_intelligence.py': '8c5fe9a17dcac5dd63d9151f73ecf305bf6e169d7e7bc69191a548b2d7435802', 'backend/app/schemas/notification.py': 'd3e6ab38d1b2b05236bad61489f9551bcdc8ecddb7882effac63a7dfce0093a1', 'backend/app/schemas/platform_integration.py': '7fecd371bc4626699ea157f9f4fd5a2a97436237012facd717b606465f2b094e', 'backend/app/schemas/project.py': '683192d08172091cd599abd985fdc39d26d6cae4026b402146690ca8316e589b', 'backend/app/schemas/query.py': '771b8ecd056a74231c7af430175b67648ff00f2898e838849dc2d5c06029c2f8', 'backend/app/schemas/report.py': '7c95a0cfd0c7b718e2f90cf531687d421d6ba6bd725a1226ca52f2b26d5c2a47', 'backend/app/schemas/translation.py': '9470aa64215d0a5fc3ebd5e9875aaa3d93339a16c645c7c461322e88f0aaaac2', 'backend/app/services/__init__.py': '5ec095bb7d0b3e1ea6a75b643b08c177c2dbcadded45e941c90e5f8b60cee893', 'backend/app/services/admin_service.py': 'bc48e23df18dff27c615bc2388ca423c23f2e4bc48bc9cdd10676eee4a3e1c97', 'backend/app/services/answer_provider.py': '86b6ad944d2925b09d716b1e7c5f85ee468e0ee5749a17c11b77654cf7dcfc96', 'backend/app/services/audit_service.py': '2972970321c96467466d827390e6b21cf16c7697284d5bdd93362cd929ee8778', 'backend/app/services/live_meeting_ingestion_service.py': '4d407d15de07ef9a23a09aa9e3666470adea103ce326ae47828c80178d69bd90', 'backend/app/services/live_session_service.py': '9e91386811c48118d3812a1b5db720708ef3b39ab4303e4a5f21f4e12f360c75', 'backend/app/services/meeting_intelligence_service.py': '9c4000a4ce8efa7431a484ab3de895e8be1bccdebb752025911371c5ac634fd7', 'backend/app/services/meeting_service.py': 'f75cb32f9f904e52cb8fa4aff87407e49270ee481e2f5c5750e5dc5d3552ddaf', 'backend/app/services/meeting_url_parser.py': '5017808ce7f3b924fe7cefbe82eb44ca884eaa68624814d83d8ff38c86ad6f21', 'backend/app/services/notification_service.py': '72da90ad6f547872166a5758988513dc08b4bfc700008263f1b2fdc85e5804f8', 'backend/app/services/platform_integration_service.py': '9d2aa6ce8eed25a1ce7433f3c17036cd31bb6569dac76185e5dc83f6775405dd', 'backend/app/services/platform_provider.py': 'b36d487c276274acec9517b09422c201aa272c2e90953a76a31cba9205f5c4bd', 'backend/app/services/project_intelligence_service.py': '00ae4986adf8ff57ac9ca39e6b7f271ee334524de2768b966816f51a582da550', 'backend/app/services/query_interface_service.py': '4524fe19625cef26558acca354a6e36876abaa95a14de1bb13992988dd111242', 'backend/app/services/report_generation_service.py': 'a908e0185d191633a1439b0833331f19bcf9de926bec9680f0b25519ab0228d0', 'backend/app/services/translation_service.py': '376004f70a5b500edc762824909cc451cc3f6c11adbfb96a0bda42d0cbfb1421', 'backend/app/skw/__init__.py': 'caf3e6538eac79bf31f856e55c65607198669543924da434a2c82345e2eb4540', 'backend/app/skw/api/__init__.py': '9bf9615b37855bd2eededcf428640c084526daf44e80f8c405433a848bc800c7', 'backend/app/skw/events/__init__.py': '0f6ecddf2cdb3c6e42e7a889ee2c6cce40ef350908a1268133e9f32d58a78e0a', 'backend/app/skw/events/contracts.py': '701cdce703660d81f38b2be932ac69781819dacb24ddb4ea7135cc23c1a97832', 'backend/app/skw/events/publisher.py': 'e55fe4488dccced3c1ed51c32f77c1b8ca13df7d3fa543e8bec27a7fa474f18f', 'backend/app/skw/indexing/__init__.py': '5d8b43e27e3d3f471ae1d51d5ec4f04470e95d2e2ef9f35460af7d427698d4e3', 'backend/app/skw/indexing/semantic_indexer.py': '609825a6a97aa4f5473706fecdcf75b57f42750405787cd1522c31eda5b5357a', 'backend/app/skw/interfaces/__init__.py': '3bbaa62b1eaed6729c66eeb5422c1393a0c0fc21e8162ae35a2fff9b449f2028', 'backend/app/skw/interfaces/component_interfaces.py': 'd14e75ffdcf71052311a5bc53a340c844839d9b3972005063a14a2af9184ce82', 'backend/app/skw/models/__init__.py': '25c082beb747a3917d3393b1153b82c2142a414eabb226be070f0a048ead0e3b', 'backend/app/skw/models/knowledge_object.py': '7d18b5979fa9ccb127bf565e6a3f204eb599f5250c466d3b7d0ec14b5e0975c5', 'backend/app/skw/repositories/__init__.py': '045b8f9e4f52a2d0fbf2360da97e102c3b46af8033bcd92f30c9f1a70b1faf8b', 'backend/app/skw/repositories/knowledge_object_repo.py': 'f4ce1be9bf30e31c9b700ef680c1cf1bac4cc3d3528deb98271981daecc6e8e7', 'backend/app/skw/schemas/__init__.py': '5d6706b1106a1b3a7855780ffeb92b71a59af4a293fa4bd5512de21b2f84a7d9', 'backend/app/skw/schemas/knowledge_object.py': '894e7b37ee93dc582011445ce6e8d255e9fb73c3b9fd04d94f9abd2acd9fbdad', 'backend/app/skw/services/__init__.py': 'e9f8fedd1037cfb98a936f3862b3ae9f8291906bd5de1cecf7e48bda1b525665', 'backend/app/skw/services/enrichment_service.py': 'd42bed420e7e2cf6f1347364bde68734f1f44f57494e0124f69cb4afd77dd405', 'backend/app/skw/services/ingestion_service.py': '07eb3791127de7eb83a30b46a186713df7e1cf129187e1ea4c6e215f7437f52c', 'backend/app/skw/services/knowledge_publisher.py': '9c6fe29a04f09e3045c45a478c2be07a9cb9a8c82ce3de0420039f86364a22f7', 'backend/app/skw/services/knowledge_query_engine.py': '05b9202f61f7661469a1c7ba4679b3ae8d0802753e6a0ddd086685563c374bbf', 'backend/app/skw/services/version_manager.py': 'ac69cf31bbb54c48aef0e841999b8e7fccf8f1b554a0fdc1b83d929cb298ea2d', 'backend/app/tests/test_frontend_api_integration_phase_4_15.py': '0fe84c7f53fce1bbcaf99dc7a19bc736895bdecd98b278183a9929b26e1af006', 'backend/cli.py': 'f7d097f1560db3beaa85f2dba0de00f0a2c33300da025e0cda787b6695f426b1', 'backend/requirements-real.txt': 'e43b48c34e4a089561341da19ce1b964076c839f0f8d6914cee59c9be5fe31cf', 'backend/requirements.txt': '1a42f40bc4747d3b927bcfe07b33e6a9239b39f58f671726e58fca0e2260f5a5', 'backend/scripts/kaggle_real_runner.py': 'cee8a76882f3f1e7d1858ecb22b8da0c3c76011731c755deb86952b2d8309a8c', 'backend/scripts/patch_pyannote_407.py': '73b05270fa8a8ab76c83d09fb5929ec2cd71b216adea31341213dc4516a0656c', 'backend/scripts/provision_real_models.py': '61c9ca3b1e582f550ec4a509af52b71c1b76a037a6f19a8367e3a464dcd52f1d', 'backend/scripts/readback_phase1.py': '95afdc1e404b367635d64e2558c590cad2c952c54a7174a3d2f21d7cd4711a60', 'backend/scripts/validate_phase1.py': '565292729ffe87e94ed4d1a4edbae7aa2febca4543732dc7500950b40670bbc4'}
BACKEND_SOURCE_ZIP_B64 = '...'
archive_bytes = base64.b64decode(BACKEND_SOURCE_ZIP_B64)
assert hashlib.sha256(archive_bytes).hexdigest() == SOURCE_ARCHIVE_SHA256
REPO.mkdir()
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
    for member in z.infolist():
        target = (REPO / member.filename).resolve()
        if not target.is_relative_to(REPO.resolve()): raise RuntimeError("Unsafe archive member")
        z.extract(member, REPO)
for relative, digest in SOURCE_MANIFEST.items():
    assert hashlib.sha256((REPO / relative).read_bytes()).hexdigest() == digest
for folder in ("runs", "artifacts", "diagnostics"):
    (REPO / "e2e_validation" / folder).mkdir(parents=True, exist_ok=True)
(REPO / "e2e_validation/execution_log.md").write_text("# Kaggle execution log\n", encoding="utf-8")
(REPO / "e2e_validation/CHECKPOINT.md").write_text("# Kaggle checkpoint\n", encoding="utf-8")
save(EVIDENCE / "source_manifest.json", {"git_commit": SOURCE_COMMIT, "working_tree_snapshot": True,
    "archive_sha256": SOURCE_ARCHIVE_SHA256, "files": SOURCE_MANIFEST})
os.environ["ABCI_SOURCE_COMMIT"] = SOURCE_COMMIT
os.environ["ABCI_SOURCE_ARCHIVE_SHA256"] = SOURCE_ARCHIVE_SHA256
print("Backend source verified:", len(SOURCE_MANIFEST), "files; commit", SOURCE_COMMIT)


## 3. Configuration and credentials
Secrets are retrieved at runtime. Original 600/30/1 settings remain in force until an actual OOM is recorded.

In [ ]:
def configure():
    try:
        from kaggle_secrets import UserSecretsClient
        secret_client = UserSecretsClient()
    except ImportError: secret_client = None
    labels = ("HF_TOKEN", "DATABASE_URL_ASYNC", "DATABASE_URL_SYNC", "REDIS_URL", "QDRANT_URL",
              "QDRANT_API_KEY", "ABCI_USER_ID", "ABCI_TENANT_ID", "ABCI_AUDIO_PATH")
    for label in labels:
        if not os.environ.get(label) and secret_client:
            try: os.environ[label] = secret_client.get_secret(label)
            except Exception: pass
    cache = WORK / "hf-cache/hub"
    cache.mkdir(parents=True, exist_ok=True)
    os.environ.update(EXECUTION_MODE="REAL", OPENMOSS_MODEL_ID="OpenMOSS-Team/MOSS-Transcribe-Diarize",
        OPENMOSS_DEVICE="cuda", OPENMOSS_CACHE_DIR=str(cache), HF_HOME=str(cache.parent),
        HF_HUB_CACHE=str(cache), HUGGINGFACE_HUB_CACHE=str(cache),
        AUDIO_CHUNK_DURATION_SECONDS="600", AUDIO_CHUNK_OVERLAP_SECONDS="30",
        AUDIO_CHUNK_THRESHOLD_SECONDS="600", AUDIO_CHUNK_CONCURRENCY="1", OPENMOSS_MAX_NEW_TOKENS="28800",
        AUDIO_CHUNK_STATE_DIR=str(WORK / "chunk-state"), AUDIO_STORAGE_PATH=str(WORK / "audio-storage"),
        DEBUG="false", ENVIRONMENT="production", LOG_LEVEL="INFO")
    os.environ.pop("TESTING", None); os.environ.pop("PYTEST_CURRENT_TEST", None)
    status = {label: "present" if os.environ.get(label) else "missing" for label in labels}
    save(EVIDENCE / "configuration_details.json", {"execution_mode": "REAL", "device": "cuda", "chunk_duration": 600,
        "overlap": 30, "concurrency": 1, "secret_presence": status})
    for label in ("HF_TOKEN", "DATABASE_URL_ASYNC", "QDRANT_URL"):
        if not os.environ.get(label): raise RuntimeError(label + " missing from environment/Kaggle Secrets")
    return {"status": "VERIFIED", "secrets": status}
gated("configuration", configure)


## 4. Hardware and compatible dependency setup
Keep Kaggle’s existing CUDA torch/torchaudio pair. A conflicting environment is BLOCKED instead of downloading replacement torch wheels. Adapter dependencies are installed into a run-specific environment; the kernel need not restart.

In [ ]:
def dependencies():
    import torch, importlib.metadata as md
    if not torch.cuda.is_available(): raise RuntimeError("Kaggle CUDA GPU unavailable; enable a GPU accelerator")
    gpu = torch.cuda.get_device_properties(0)
    torch_version = md.version("torch"); audio_version = md.version("torchaudio")
    if torch_version.split("+")[0] != audio_version.split("+")[0]:
        raise RuntimeError("Existing torch/torchaudio versions differ; use a compatible Kaggle runtime")
    save(EVIDENCE / "hardware.json", {"gpu": gpu.name, "vram_bytes": gpu.total_memory,
        "cuda": torch.version.cuda, "torch": torch_version, "torchaudio": audio_version,
        "python": sys.version.split()[0]})
    venv = WORK / "venv"
    proc = subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", str(venv)], text=True, capture_output=True)
    if proc.returncode: raise RuntimeError(redact(proc.stdout + proc.stderr))
    python = venv / "bin/python"
    proc = subprocess.run([sys.executable, "-m", "pip", "--python", str(python),
        "install", "--disable-pip-version-check", "pip", "wrapt"], text=True, capture_output=True)
    save(EVIDENCE / "venv_bootstrap.json", {"exit_code": proc.returncode,
        "stdout": redact(proc.stdout), "stderr": redact(proc.stderr)})
    if proc.returncode: raise RuntimeError("Pip bootstrap failed: " + redact(proc.stdout + proc.stderr))
    constraints = WORK / "torch-constraints.txt"
    constraints.write_text(f"torch=={torch_version}\ntorchaudio=={audio_version}\n")
    requirements = ["transformers==5.16.1", "accelerate==1.14.0", "safetensors==0.8.0",
        "bitsandbytes==0.50.2", "pyannote.audio==4.0.7", "av==17.1.0", "librosa>=0.11,<0.12",
        "soundfile>=0.13,<0.15", "soxr>=1,<2"]
    commands = [[str(python), "-m", "pip", "install", "--disable-pip-version-check", "-c", str(constraints),
                 "-r", str(REPO / "backend/requirements.txt"), *requirements],
                [str(python), "-m", "pip", "install", "--no-deps", "--disable-pip-version-check",
                 "moss-transcribe-diarize @ git+https://github.com/OpenMOSS/MOSS-Transcribe-Diarize.git@61bc29cd4120be7b5d3b761b64cd5dff57263642"]]
    for index, command in enumerate(commands):
        proc = subprocess.run(command, text=True, capture_output=True)
        (EVIDENCE / f"dependency_{index}.log").write_text(redact(proc.stdout + proc.stderr))
        if proc.returncode: raise RuntimeError("Compatible dependency installation failed; see saved dependency log (torch unchanged)")
    # Isolate the reproducible PyAnnote package patch inside this run's venv.
    overlay_script = """from importlib.metadata import distribution
from pathlib import Path
import shutil,sysconfig
d=distribution('pyannote.audio'); site=Path(sysconfig.get_paths()['purelib'])
for file in d.files:
 p=Path(str(file))
 if p.parts[0]=='pyannote' and len(p.parts)>1 and p.parts[1]=='audio' or p.parts[0].endswith('.dist-info'):
  src=Path(d.locate_file(file)); dst=site/p
  if src.is_file() and src.resolve()!=dst.resolve():
   dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
"""
    proc = subprocess.run([str(python), "-c", overlay_script], capture_output=True, text=True)
    if proc.returncode: raise RuntimeError("Could not isolate PyAnnote compatibility package")
    proc = subprocess.run([str(python), str(REPO / "backend/scripts/patch_pyannote_407.py")], capture_output=True, text=True)
    (EVIDENCE / "pyannote_compatibility.log").write_text(redact(proc.stdout + proc.stderr))
    if proc.returncode: raise RuntimeError("Version-checked PyAnnote compatibility patch failed")
    os.environ["ABCI_PYTHON"] = str(python)
    return {"status": "VERIFIED", "torch_reused": True, "gpu": gpu.name}
gated("dependencies", dependencies, ("configuration",))


## 5. Service connectivity
Use the existing PostgreSQL, Redis, Qdrant and authorized SKW paths. No fake services, SQLite fallback or in-memory Qdrant. Database migrations are an external prerequisite; this notebook never performs destructive schema operations.

In [ ]:
def worker(stage, folder, extra=()):
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    existing = folder / f"{stage}_stage.json"
    if existing.exists(): return json.loads(existing.read_text())
    cmd = [os.environ["ABCI_PYTHON"], str(REPO / "backend/scripts/kaggle_real_runner.py"),
           stage, "--run", str(folder), *map(str, extra)]
    started = time.perf_counter()
    proc = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
    save(folder / f"{stage}_process.json", {"exit_code": proc.returncode, "elapsed_seconds": time.perf_counter()-started})
    (folder / f"{stage}_launcher.log").write_text(redact(proc.stdout + proc.stderr))
    if existing.exists(): return json.loads(existing.read_text())
    return {"status": "BLOCKED", "reason": "Worker failed before a structured result; see saved launcher log"}
gated("diagnostics", lambda: worker("diagnostics", EVIDENCE / "diagnostics"), ("dependencies",))
gated("services", lambda: worker("services", EVIDENCE / "services"), ("diagnostics",))


## 6. Local model cache reuse and complete weight provisioning
Only required incomplete model repositories are provisioned. Cached snapshots are linked into a writable overlay; existing complete weights are never fetched again. HF gate failures remain BLOCKED.

In [ ]:
def models():
    cache = Path(os.environ["HF_HUB_CACHE"])
    mounted = Path("/kaggle/input")
    for source in mounted.rglob("models--*"):
        if not source.is_dir() or source.parent.name != "hub": continue
        for item in source.rglob("*"):
            if not item.is_file() or ".locks" in item.parts: continue
            target = cache / source.name / item.relative_to(source)
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.symlink_to(item.resolve())
    check_code = "import sys,json;sys.path.insert(0,'backend');from app.ai.model_inventory import inventory;print(json.dumps(inventory()))"
    proc = subprocess.run([os.environ["ABCI_PYTHON"], "-c", check_code], cwd=REPO, capture_output=True, text=True)
    if proc.returncode: raise RuntimeError("Local model inventory failed")
    inventory = json.loads(proc.stdout)
    save(EVIDENCE / "model_inventory_before.json", inventory)
    missing = [row["model"] for row in inventory if row["required"] and row["status"] != "COMPLETE"]
    if missing:
        cmd = [os.environ["ABCI_PYTHON"], str(REPO / "backend/scripts/provision_real_models.py"),
               "--models", *missing, "--report", str(EVIDENCE / "model_provisioning.json")]
        plan_cmd = cmd + ["--dry-run"]
        # Use distinct report paths so the plan is retained.
        plan_cmd[plan_cmd.index("--report")+1] = str(EVIDENCE / "model_provisioning_plan.json")
        proc = subprocess.run(plan_cmd, cwd=REPO, text=True, capture_output=True)
        (EVIDENCE / "model_plan.log").write_text(redact(proc.stdout + proc.stderr))
        if proc.returncode: raise RuntimeError("Official cache provisioning plan blocked; check HF access and Internet")
        plan = json.loads((EVIDENCE / "model_provisioning_plan.json").read_text())
        print("Missing required model repositories:", missing)
        print("Missing artifacts only; complete cached equivalents are reused:")
        for entry in plan["models"]:
            print(json.dumps({"model": entry["model"], "download_bytes": entry["download_bytes"],
                "reuse_bytes": entry["reuse_bytes"], "destination": plan["cache"],
                "reason": "Required local snapshot incomplete"}))
        print("Download size/destination/reuse plan saved:", str(EVIDENCE / "model_provisioning_plan.json"))
        proc = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
        (EVIDENCE / "model_provisioning.log").write_text(redact(proc.stdout + proc.stderr))
        if proc.returncode: raise RuntimeError("Required weights unavailable; see provisioning evidence; no substituted models")
    proc = subprocess.run([os.environ["ABCI_PYTHON"], "-c", check_code], cwd=REPO, capture_output=True, text=True)
    if proc.returncode: raise RuntimeError("Post-provisioning inventory failed")
    complete = json.loads(proc.stdout); save(EVIDENCE / "model_status.json", complete)
    if any(row["required"] and row["status"]!="COMPLETE" for row in complete):
        raise RuntimeError("Required model weights/shards incomplete")
    os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"
    return {"status": "VERIFIED", "artifacts_complete": True, "inference": "NOT VERIFIED"}
gated("models", models, ("services",))


## 7–8. Actual audio selection and small REAL inference
The smoke uses a genuine interval; it is never the final result. Audio metadata/hash are measured from the complete input file.

In [ ]:
AUDIO = None
def select_audio():
    global AUDIO
    explicit = os.environ.get("ABCI_AUDIO_PATH")
    paths = [Path(explicit)] if explicit else sorted(p for p in Path("/kaggle/input").rglob("*")
        if p.is_file() and p.suffix.lower() in (".wav", ".mp3", ".flac", ".m4a") and not any(x.startswith("models--") for x in p.parts))
    preferred = [p for p in paths if p.name == "ES2002a.Mix-Headset.wav"]
    if preferred: paths = preferred
    if len(paths)!=1 or not paths[0].is_file():
        raise RuntimeError("Attach one actual full audio file or set ABCI_AUDIO_PATH to the intended mounted input")
    AUDIO = paths[0].resolve()
    return {"status": "VERIFIED", "audio_filename": AUDIO.name}
gated("audio", select_audio, ("models",))
def smoke():
    start = 71 if AUDIO.name == "ES2014a.Mix-Headset.wav" else 0
    result = worker("probe", EVIDENCE / "real_smoke", ("--audio", AUDIO, "--duration", 10, "--start", start, "--all-models"))
    result["scope"] = "10-second actual provider smoke, not full production"
    return result
gated("real_smoke", smoke, ("audio",))


## 9. Evidence-based configured chunk probe / OOM handling

First attempt the configured 600-second interval through the existing MOSS
provider. An actual CUDA OOM permits distinct smaller candidates; each executes
in a fresh process to release failed CUDA allocations. No identical failed size
is retried. Non-OOM failures stop the gate. Full audio is never truncated.
Working overlap is 30 s when valid, otherwise one fifth of chunk duration.
Both original/working settings and the reason are recorded.

In [ ]:
def choose_chunks():
    attempts = []
    for duration in (600, 300, 120, 60, 30, 15, 10):
        os.environ["OPENMOSS_MAX_NEW_TOKENS"] = str(max(5120, duration * 48))
        folder = EVIDENCE / f"chunk_probe_{duration}"
        result = worker("probe", folder, ("--audio", AUDIO, "--duration", duration))
        attempts.append({"duration": duration, "result": result, "evidence": str(folder)})
        if result["status"] == "VERIFIED":
            overlap = 30 if duration>30 else duration/5
            os.environ.update(AUDIO_CHUNK_DURATION_SECONDS=str(duration), AUDIO_CHUNK_OVERLAP_SECONDS=str(overlap),
                              AUDIO_CHUNK_THRESHOLD_SECONDS=str(duration), AUDIO_CHUNK_CONCURRENCY="1")
            choice = {"status": "VERIFIED", "configured_duration": 600, "configured_overlap": 30,
                "working_duration": duration, "working_overlap": overlap, "step": duration-overlap,
                "reason": "Configured size passed" if duration==600 else "Smaller distinct interval passed after recorded CUDA OOM",
                "attempts": attempts}
            save(EVIDENCE / "chunk_configuration.json", choice)
            print("Working chunk configuration:", duration, "seconds; overlap", overlap)
            return choice
        probe_file = folder / "probe_result.json"
        probe = json.loads(probe_file.read_text()) if probe_file.exists() else {}
        if not probe.get("oom"):
            save(EVIDENCE / "chunk_configuration.json", {"status":"BLOCKED", "attempts":attempts})
            return {"status": "BLOCKED", "reason": "Non-OOM inference failure; no configuration workaround or fallback"}
    save(EVIDENCE / "chunk_configuration.json", {"status":"BLOCKED", "attempts":attempts})
    return {"status":"BLOCKED", "reason":"No tested chunk size completed real MOSS generation"}
gated("working_chunks", choose_chunks, ("real_smoke",))


## 10–14. Full REAL production and independent persistence read-back

The **entire original input** is passed to the existing CLI. LongAudioProcessor
plans and processes all chunks. Actual MOSS, PyAnnote, Sarvam, ACE, Blackboard,
SKW, PostgreSQL, Redis and Qdrant evidence is captured by the repository recorder.
PyAnnote retains its existing full-file pipeline; its actual observed global turns
are additionally grouped into production chunk windows, not relabelled as separate
chunk inference. Original ASR text remains canonical; normalization accuracy is unknown.

State journals retain actual completed global chunk segments, SHA-256/model/config
identity and IDs. A later explicit `ABCI_RESUME_MEETING_ID` may select the same
committed meeting via CLI; this notebook does not automatically exercise resume
or claim it verified. Existing overlap source separation remains unconfigured:
if the actual module blocks on overlapping speech, the run is BLOCKED and saved.
No fake separation/semantic output is inserted.

In [ ]:
FULL_RUN = REPO / "e2e_validation/runs" / RUN_ID
def full_production():
    extra = ["--audio", AUDIO]
    resume = os.environ.get("ABCI_RESUME_MEETING_ID")
    if resume: extra += ["--resume-meeting-id", resume]
    return worker("production", FULL_RUN, extra)
gated("full_production", full_production, ("working_chunks",))
def persistence_readback():
    if not (FULL_RUN / "execution.json").exists():
        return {"status":"BLOCKED", "reason":"No production meeting execution to read back"}
    return worker("readback", FULL_RUN)
# Read back actual storage even after a failed production attempt; do not hide absence.
gated("fresh_process_readback", persistence_readback, ("dependencies", "services"))


## 15–17. Measured performance, evidence and final verification
VERIFIED requires the entire supplied file to have actual successful chunk inference, valid reconstruction, committed canonical text, fresh-process matching read-back and existing CLI success. No ground-truth accuracy metrics are invented.

In [ ]:
def final_verification():
    if not (FULL_RUN / "execution.json").exists():
        return {"status":"BLOCKED", "reason":"Full production never executed"}
    return worker("report", FULL_RUN)
gated("final_verification", final_verification, ("dependencies",))
actual_path = FULL_RUN / "final_verification.json"
actual = json.loads(actual_path.read_text()) if actual_path.exists() else {}
hardware_path = EVIDENCE / "hardware.json"
hardware = json.loads(hardware_path.read_text()) if hardware_path.exists() else {}
ids = actual.get("IDS", {})
status = actual.get("STATUS", "BLOCKED")
if not actual.get("gates", {}).get("entire_audio_coverage", False): status = "BLOCKED"
final = {
    "STATUS": status, "GPU / VRAM": hardware,
    "AUDIO FILE": str(AUDIO) if AUDIO else "NOT AVAILABLE", "AUDIO DURATION": actual.get("AUDIO", {}).get("audio_duration", "NOT AVAILABLE"),
    "CHUNK CONFIGURATION": STAGES.get("working_chunks", "NOT VERIFIED"), "CHUNK COUNT": len(actual.get("CHUNKS", [])),
    "COVERAGE": actual.get("gates", {}).get("entire_audio_coverage", False), "GAPS": "See measured chunk plan; no lossless claim",
    "MOSS": actual.get("MOSS", "NOT VERIFIED"), "SARVAM": actual.get("SARVAM", "NOT VERIFIED"),
    "PYANNOTE": actual.get("PYANNOTE", "NOT VERIFIED"),
    "SPEAKER RECONCILIATION": actual.get("gates", {}).get("speaker_reconciliation", False),
    "TIMESTAMP RECONSTRUCTION": actual.get("gates", {}).get("timestamps", False),
    "DEDUPLICATION": actual.get("gates", {}).get("deduplication_executed", False),
    "ACE": actual.get("gates", {}).get("ace", False), "BLACKBOARD": "Actual saved task/event artifacts; no synthetic status",
    "SKW": "See actual committed object and Qdrant read-back artifacts" if actual else "NOT VERIFIED",
    "POSTGRESQL": actual.get("POSTGRESQL", "NOT VERIFIED"), "FRESH READ-BACK": actual.get("FRESH_READ_BACK", "NOT VERIFIED"),
    "REDIS": actual.get("REDIS", "NOT VERIFIED"), "QDRANT": actual.get("QDRANT", "NOT VERIFIED"),
    "CLI": actual.get("gates", {}).get("cli", False), "PROCESSING TIME": actual.get("PROCESSING_TIME", "NOT AVAILABLE"),
    "RTF": actual.get("RTF", "NOT AVAILABLE"), "MEETING ID": ids.get("meeting_id", "NOT AVAILABLE"),
    "TRANSCRIPT ID": ids.get("transcript_id", "NOT AVAILABLE"), "PROJECT ID": ids.get("project_id", "NOT AVAILABLE"),
    "CORRELATION ID": ids.get("correlation_ids", "NOT AVAILABLE"), "ARTIFACTS": str(WORK),
    "FAILURES": actual.get("FAILURES", BLOCKERS), "RETRIES": actual.get("RETRIES", "NOT AVAILABLE"),
    "REMAINING GAPS": actual.get("REMAINING_GAPS", ["Full production did not complete"]),
    "Resume/retry verification": "NOT VERIFIED unless actually exercised",
    "Semantic retrieval accuracy": "NOT VERIFIED", "ASR/speaker accuracy": "NOT VERIFIED without ground truth",
    "SOURCE COMMIT": SOURCE_COMMIT, "SOURCE SNAPSHOT SHA-256": SOURCE_ARCHIVE_SHA256,
}
save(EVIDENCE / "final_report.json", final)
save(EVIDENCE / "run_metadata.json", {"run_id": RUN_ID, "timestamp": datetime.now(timezone.utc).isoformat(),
    "git_commit": SOURCE_COMMIT, "source_archive_sha256": SOURCE_ARCHIVE_SHA256, "stages": STAGES})
with (REPO / "e2e_validation/execution_log.md").open("a", encoding="utf-8") as log:
    log.write("\n## Notebook final report\n```json\n" + json.dumps(final, indent=2) + "\n```\n")
with (REPO / "e2e_validation/CHECKPOINT.md").open("a", encoding="utf-8") as checkpoint:
    checkpoint.write(f"\nRun {RUN_ID}: {status}. Evidence: {EVIDENCE}. No accuracy or resume claim.\n")
for name in ("execution_log.md", "CHECKPOINT.md"):
    shutil.copy2(REPO / "e2e_validation" / name, EVIDENCE / name)
shutil.make_archive(str(WORK / "execution_evidence"), "zip", root_dir=WORK,
    base_dir="evidence")
if FULL_RUN.exists(): shutil.make_archive(str(WORK / "production_evidence"), "zip", root_dir=FULL_RUN)
if (WORK / "chunk-state").exists(): shutil.make_archive(str(WORK / "chunk_state"), "zip", root_dir=WORK / "chunk-state")
# Display structural evidence only; raw transcript/model/SQL payloads stay in artifacts.
display({key: value for key, value in final.items() if key not in ("FAILURES", "CHUNK CONFIGURATION")})
print("Overall:", status)
print("Evidence directory:", WORK)


### Interpretation and references

- A model inventory, successful import, HTTP 200 or empty artifact is not inference success.
- Timeline coverage refers to processing all real audio chunks, not proof that every word was transcribed correctly.
- Speaker mappings and overlap dedup decisions are observed algorithm outputs, not ground-truth accuracy.
- Detected overlap without a separation provider is explicitly UNRESOLVED; original ASR text is retained
  with persisted quality warnings. No separated audio, corrected overlap words or separation accuracy is claimed.
- Unavailable services, gated model access, dependency conflicts,
  token-limit failures or model OOMs are recorded as actual blockers.
- Do not publish credentials or private transcript evidence with notebook output. Source code contains
  no local environment file; executed artifacts contain actual user audio/transcript data.
- Official sources: [OpenMOSS inference/API](https://github.com/OpenMOSS/MOSS-Transcribe-Diarize),
  [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md),
  [Sarvam base model](https://huggingface.co/sarvamai/sarvam-1).